# master

In [54]:
%reset -f

In [55]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image

PL.reset()

In [56]:
ol = Overlay('master-zcu102.bit')

In [57]:
#help(ol)

In [58]:
c2c = ol.axi_chip2chip_0

In [59]:
#type(c2c)

In [60]:
#help(c2c)

In [61]:
val = c2c.read(0x00)

# Print in hex
print(f"Read: {val:#x}")

Read: 0x4


In [62]:
# Start the IP core by setting the ap_start bit in CTRL register
c2c.write(0x00, 0x1)

In [67]:
# Read back the  contol signals at offset 0x00
val = c2c.read(0x00)

# Print in hex
print(f"Read: {val:#x}")

Read: 0x1


In [68]:
from PIL import Image
def save_img(fname, tensor):
    img_tensor = np.squeeze(tensor,axis=2)  # Remove batch dim → [C, H, W]
    img = Image.fromarray(img_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
    save_path=f"{fname}.png"
    img.save(save_path)
    return save_path

In [69]:
def unpack_frame(packed):
    H, W_packed,ch = packed.shape  # (480, 160,4)
    W = W_packed * ch            # Unpacked width = 640
    output_tensor = np.reshape(frame, (H, W, 1))
    return output_tensor

In [70]:
from pynq import MMIO
import numpy as np
import struct
from datetime import datetime

# Frame parameters
height, width, channels = 480, 160, 4
num_bytes = height * width * channels  # = 307200 bytes
num_words = num_bytes // 4  # = 76800  (32-bit words)


# Memory base address (PL side)
C2C_BASE_ADDR_OUTPUT_VALUE = 0x1F000000

# Initialize MMIO
mmio = MMIO(C2C_BASE_ADDR_OUTPUT_VALUE, 0x80000)

# Read 32-bit words and unpack into bytes
raw_bytes = bytearray()
for i in range(num_words):
    word = mmio.read(i * 4)              # Read 32-bit word
    raw_bytes.extend(struct.pack("<I", word))  # Little-endian unpacking into 4 bytes

# Convert to numpy array and reshape to image
frame = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((height, width, channels))
unpacked_frame = unpack_frame(frame)
print(f"type(unpacked_frame)={type(unpacked_frame)},\nunpacked_frame.shape={unpacked_frame.shape},\nunpacked_frame.dtype={unpacked_frame.dtype}")


timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
fname=f"{timestamp}-image-output"
#fname=f"image-output"
save_img(fname=fname, tensor=unpacked_frame)


type(unpacked_frame)=<class 'numpy.ndarray'>,
unpacked_frame.shape=(480, 640, 1),
unpacked_frame.dtype=uint8


'2022-10-22_05-44-43-image-output.png'

In [72]:
from pynq import MMIO

TARGET_ADDR = 0x1F000000
mmio = MMIO(TARGET_ADDR, 4)  # Map 4 bytes (1 word)

mmio.write(0, 0xDEADBEEF)  # Write value at offset 0 (TARGET_ADDR)

val = mmio.read(0)
print(f"Read back value: 0x{val:08X}")

Read back value: 0xDEADBEEF


In [ ]:
import mmap, os, numpy as np

FRAME_ADDR = 0x1F000000  # must match Vivado + C2C slave's target
FRAME_SIZE = 102 * 160 * 4  # 65520 bytes

fd = os.open("/dev/mem", os.O_RDWR | os.O_SYNC)
mem = mmap.mmap(fd, FRAME_SIZE, mmap.MAP_SHARED, mmap.PROT_READ, offset=FRAME_ADDR)

frame_data = np.frombuffer(mem, dtype=np.uint8).reshape((102, 160, 4))

mem.close()
os.close(fd)

In [19]:
from pynq import MMIO
import time

# Define addresses
C2C_BASE_ADDR_INIT              = 0xA0020000  # Replace with actual address
#C2C_BASE_ADDR_INPUT_VALUE       = 0xA0000010
#C2C_BASE_ADDR_OUTPUT_OFFSET     = 0xA0000018
C2C_BASE_ADDR_OUTPUT_VALUE      = 0xFFFC0004

# Register offsets
XEXECUTE_CFG_PORT_ADDR_AP_CTRL            = 0x00
XEXECUTE_CFG_PORT_ADDR_GIE                = 0x04
XEXECUTE_CFG_PORT_ADDR_IER                = 0x08
XEXECUTE_CFG_PORT_ADDR_ISR                = 0x0c
XEXECUTE_CFG_PORT_ADDR_DATA_PORT_DATA     = 0x10
XEXECUTE_CFG_PORT_BITS_DATA_PORT_DATA     = 32
XEXECUTE_CFG_PORT_ADDR_FRAME_CNT_DATA     = 0x18
XEXECUTE_CFG_PORT_BITS_FRAME_CNT_DATA     = 32
XEXECUTE_CFG_PORT_ADDR_END_OF_STREAM_DATA = 0x20
XEXECUTE_CFG_PORT_BITS_END_OF_STREAM_DATA = 1

#XEXECUTE_CFG_PORT1_ADDR_DATA_PORT_DATA = 0x28

# Initialize MMIO (assuming 64KB mapping region size)
mmio_c2c = MMIO(C2C_BASE_ADDR_INIT, 0x10000)
mmio_output = MMIO(C2C_BASE_ADDR_OUTPUT_VALUE, 0x10000)

# Set values
init_value = 0x00000001
#test_value = 0x00000042

# Write address low to DATA_PORT_DATA
addr_low = C2C_BASE_ADDR_OUTPUT_VALUE & 0xFFFFFFFF
mmio_c2c.write(XEXECUTE_CFG_PORT_ADDR_DATA_PORT_DATA, addr_low)

# Confirm address write
read_back = mmio_c2c.read(XEXECUTE_CFG_PORT_ADDR_DATA_PORT_DATA)
if read_back != addr_low:
    print("Bad address low!!!!")
print(f"0x{read_back:08X}")

# Test output memory mapping
mmio_output.write(0, 0xDEADBEEF)
if mmio_output.read(0) != 0xDEADBEEF:
    print("Wrong memory map!!!!!!!!!!")
else:
    print("Successfully write/read memory!")

# Write test input value
#print("Trying to write input...")
#mmio_input = MMIO(C2C_BASE_ADDR_INPUT_VALUE, 0x10000)
#mmio_input.write(0, test_value)
#_ = mmio_input.read(0)  # Dummy read
#time.sleep(0.01)        # Small delay
#confirm_write = mmio_input.read(0)
#print(f"Confirm write val: 0x{confirm_write:08X}")

# Write init config value
print("Trying to initialise...")
mmio_c2c.write(0, init_value)
_ = mmio_c2c.read(0)  # Dummy read
time.sleep(0.01)      # Small delay
confirm_init = mmio_c2c.read(0)
print(f"Confirm init val: 0x{confirm_init:08X}")

# Read back result
read_value = mmio_output.read(0)
print(f"HLS Output = 0x{read_value:08X}")
if read_value == (0x40 + test_value):
    print("You rock! :)")
else:
    print("You suck! :(")


0xFFFC0004
Successfully write/read memory!
Trying to initialise...
Confirm init val: 0x00000001
HLS Output = 0xDEADBEEF


NameError: name 'test_value' is not defined

# --------------------------

In [5]:
img2axis = ol.img2axis_0

In [6]:
# help(img2axis.register_map)

In [7]:
def start_img_to_axis(ip,buffer, eos,frame_cnt):
    
# Configure registers:
   #c2c.write(img_2_axis_base,1)
   # ip.register_map.CTRL.AP_START=1

In [8]:
def unpack_frame(packed):
    H, W_packed,ch = packed.shape  # (480, 160,4)
    W = W_packed * ch            # Unpacked width = 640
    output_tensor = np.reshape(frame, (H, W, 1))
    return output_tensor


In [9]:
def save_img(fname, tensor):
    img_tensor = np.squeeze(tensor,axis=2)  # Remove batch dim → [C, H, W]
    img = Image.fromarray(img_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
    save_path=f"{fname}.png"
    img.save(save_path)
    return save_path

In [40]:
import socket
import numpy as np
import time  # for simulating delay between frames

def send_frame(unpacked_frame, DEST_IP = '192.168.100.119',DEST_PORT = 5005):

    
    # Remove the singleton channel dimension (shape becomes 480x640)
    frame_bytes = unpacked_frame.squeeze(axis=2).tobytes()

    # Create a TCP socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect((DEST_IP, DEST_PORT))

    # Optionally, send the frame size first (so the receiver knows what to expect)
    frame_size = len(frame_bytes)
    sock.sendall(frame_size.to_bytes(4, byteorder='big'))  # Send 4-byte length

    # Send the frame
    sock.sendall(frame_bytes)

    # Close the connection
    sock.close()

    print("Frame sent successfully.")

In [35]:
#img_to_axis(ol.img2axis_0,buff_o,True,4)
#start_img_to_axis

In [38]:
unpacked_frame = unpack_frame(frame)
print(f"type(unpacked_frame)={type(unpacked_frame)},\nunpacked_frame.shape={unpacked_frame.shape},\nunpacked_frame.dtype={unpacked_frame.dtype}")

type(unpacked_frame)=<class 'pynq.buffer.PynqBuffer'>,
unpacked_frame.shape=(480, 640, 1),
unpacked_frame.dtype=uint8


In [73]:
send_frame(unpacked_frame)

Frame sent successfully.


In [45]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
#fname=f"{timestamp}-image-output"
fname=f"image-output"
save_img(fname=fname, tensor=unpacked_frame)

'image-output.png'

In [ ]:
from pynq import MMIO
import time

# Define addresses
C2C_BASE_ADDR_INIT              = 0xA0000000  # Replace with actual address
C2C_BASE_ADDR_INPUT_VALUE       = 0xA0000010
C2C_BASE_ADDR_OUTPUT_OFFSET     = 0xA0000018
C2C_BASE_ADDR_OUTPUT_VALUE      = 0xFFFC0004

# Register offsets
XEXECUTE_CFG_PORT1_ADDR_DATA_PORT_DATA = 0x28

# Initialize MMIO (assuming 4KB mapping region size)
mmio_c2c = MMIO(C2C_BASE_ADDR_INIT, 0x100)
mmio_output = MMIO(C2C_BASE_ADDR_OUTPUT_VALUE, 0x100)

# Set values
init_value = 0x00000003
test_value = 0x00000042

# Write address low to DATA_PORT_DATA
addr_low = C2C_BASE_ADDR_OUTPUT_VALUE & 0xFFFFFFFF
mmio_c2c.write(XEXECUTE_CFG_PORT1_ADDR_DATA_PORT_DATA, addr_low)

# Confirm address write
read_back = mmio_c2c.read(XEXECUTE_CFG_PORT1_ADDR_DATA_PORT_DATA)
if read_back != addr_low:
    print("Bad address low!!!!")
print(f"0x{read_back:08X}")

# Test output memory mapping
mmio_output.write(0, 0xDEADBEEF)
if mmio_output.read(0) != 0xDEADBEEF:
    print("Wrong memory map!!!!!!!!!!")
else:
    print("Successfully write/read memory!")

# Write test input value
print("Trying to write input...")
mmio_input = MMIO(C2C_BASE_ADDR_INPUT_VALUE, 0x100)
mmio_input.write(0, test_value)
_ = mmio_input.read(0)  # Dummy read
time.sleep(0.01)        # Small delay
confirm_write = mmio_input.read(0)
print(f"Confirm write val: 0x{confirm_write:08X}")

# Write init value
print("Trying to initialise...")
mmio_c2c.write(0, init_value)
_ = mmio_c2c.read(0)  # Dummy read
time.sleep(0.01)      # Small delay
confirm_init = mmio_c2c.read(0)
print(f"Confirm init val: 0x{confirm_init:08X}")

# Read back result
read_value = mmio_output.read(0)
print(f"HLS Output = 0x{read_value:08X}")
if read_value == (0x40 + test_value):
    print("You rock! :)")
else:
    print("You suck! :(")


# playground

In [ ]:
hasattr(ol.axi_vdma_0, 'write')  # should return True


In [ ]:
img_to_axis(ol.img2axis_0,buff_o,True,88)

In [ ]:
help(VideoMode)

In [ ]:
dir(ol.axi_vdma_0)

In [ ]:
ol.axi_vdma_0.framecount

In [ ]:
len(ol.axi_vdma_0.readchannel._frames)